# Style Prefix Demo (GPT-2 Nano)
This notebook installs Levanter from the `feat/style-prefix-token` branch,
creates a tiny chat dataset with `style` labels, inspects the resulting
prefix tokens and loss mask, and runs a short GPT-2 "nano" training loop.

In [ ]:
%%bash

set -euo pipefail

echo 'Installing dependencies (levanter feat/style-prefix-token, datasets, wandb, draccus, jax, jaxlib)...'
pip install --quiet \
  git+https://github.com/chris544460/levanter.git@feat/style-prefix-token \
  datasets wandb draccus jax jaxlib
echo 'Dependencies installed.
'

REPO_DIR=/content/levanter

if [ -d "$REPO_DIR/.git" ]; then
  echo "Found existing repo at $REPO_DIR. Syncing feat/style-prefix-token...
"
  git -C "$REPO_DIR" fetch origin feat/style-prefix-token
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  echo 'Resetting local branch to origin/feat/style-prefix-token...'
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
  echo 'Cleaning untracked files...'
  git -C "$REPO_DIR" clean -fd
else
  echo "No repo found. Cloning fresh copy to $REPO_DIR...
"
  rm -rf "$REPO_DIR"
  git clone https://github.com/chris544460/levanter.git "$REPO_DIR"
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
fi

echo 'Ensuring style demo dataset exists...'
mkdir -p "$REPO_DIR/data/style_demo"
cat <<'EOF' > "$REPO_DIR/data/style_demo/train.jsonl"
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "Who wrote the Odyssey?"}, {"role": "assistant", "content": "Homer wrote the Odyssey."}], "style": "wiki"}
{"messages": [{"role": "system", "content": "style=books"}, {"role": "user", "content": "Recommend a fantasy series."}, {"role": "assistant", "content": "Try The Wheel of Time by Robert Jordan."}], "style": "books"}
{"messages": [{"role": "system", "content": "style=news"}, {"role": "user", "content": "Summarize today's headlines."}, {"role": "assistant", "content": "Major markets rallied; new policies were announced."}], "style": "news"}
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "What is photosynthesis?"}, {"role": "assistant", "content": "Photosynthesis converts light energy into chemical energy."}], "style": "wiki"}
EOF
echo 'Style demo dataset written to $REPO_DIR/data/style_demo/train.jsonl.'



In [ ]:
%%bash
cd /content/levanter
pip uninstall -y levanter

pip install -e .

In [ ]:
import sys
from pathlib import Path

_path_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
for _candidate in _path_candidates:
    _src = (_candidate / 'src').resolve()
    if _src.exists():
        if str(_src) not in sys.path:
            sys.path.append(str(_src))
        break
else:
    raise ModuleNotFoundError('Could not locate Levanter src directory. Add repo path to sys.path manually.')

from levanter.data.text import ChatLmDatasetFormat, StylePrefixConfig, preprocessor_for_format

chat_template = (
    "{%- for message in messages -%}\n"
    "{{ message['role'] }}: {{ message['content'] }}\n"
    "{%- endfor -%}\n"
    "{%- if add_generation_prompt %}assistant:{% endif %}"
)

format_cfg = ChatLmDatasetFormat(
    messages_field="messages",
    single_turn=False,
    chat_template=chat_template,
    pack=True,
    mask_user_turns=False,
    style_prefix=StylePrefixConfig(
        prefix_token="<style>",
        suffix_token="</style>",
        style_field="style",
    ),
)

processor = preprocessor_for_format(format_cfg, tokenizer)
processed = processor(entries[:2])

for idx, example in enumerate(processed):
    print(f"Example {idx}")
    print("input_ids:", example["input_ids"][:20])
    print("assistant_masks:", example["assistant_masks"][:20])
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"][:20])
    print("tokens:", tokens)
    print()


In [ ]:
%%bash
cd /content/levanter
PYTHONPATH=$PWD python -m levanter.main.train_lm --config_path config/gpt2_nano_style.yaml